# 📊 Notebook 01 — EDA & Yield Analysis
## Fab Intelligence: Semiconductor Manufacturing Analytics

**Objective:** Understand the baseline yield, data quality, and failure distribution of the SECOM semiconductor manufacturing dataset.

**Business Question:** *What is our current yield rate, where are the data gaps, and what does the failure profile look like over time?*

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'figure.facecolor': '#F8FAFC'
})
BLUE   = '#1A3C6E'
RED    = '#C0392B'
GREEN  = '#27AE60'
ORANGE = '#E67E22'
GRAY   = '#95A5A6'

print('Libraries loaded.')

In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────────
df = pd.read_csv('../data/secom_combined.csv')
labels = pd.read_csv('../data/secom_labels.csv')

# Sensor columns only (exclude label columns)
sensor_cols = [c for c in df.columns if c not in ['label', 'label_text']]

print(f'Dataset shape     : {df.shape}')
print(f'Sensor columns    : {len(sensor_cols)}')
print(f'Total wafers      : {len(df):,}')
print(f'PASS wafers       : {(df.label == -1).sum():,} ({(df.label == -1).mean()*100:.1f}%)')
print(f'FAIL wafers       : {(df.label ==  1).sum():,} ({(df.label ==  1).mean()*100:.1f}%)')
print(f'Missing values    : {df[sensor_cols].isna().sum().sum():,} ({df[sensor_cols].isna().mean().mean()*100:.1f}% of sensor data)')

In [ ]:
# ── Figure 1: Yield Overview Dashboard ────────────────────────────────────
fig = plt.figure(figsize=(16, 10), facecolor='#F8FAFC')
fig.suptitle('Semiconductor Fab — Yield Overview Dashboard', 
             fontsize=18, fontweight='bold', color=BLUE, y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.4)

# 1a. Yield Donut
ax1 = fig.add_subplot(gs[0, 0])
pass_count = (df.label == -1).sum()
fail_count = (df.label ==  1).sum()
wedges, texts, autotexts = ax1.pie(
    [pass_count, fail_count],
    labels=['PASS', 'FAIL'],
    colors=[GREEN, RED],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'width': 0.5},
    textprops={'fontsize': 11}
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
ax1.set_title('Overall Yield Rate', fontweight='bold', color=BLUE)
ax1.text(0, 0, f'{pass_count/len(df)*100:.1f}%\nYield', 
         ha='center', va='center', fontsize=13, fontweight='bold', color=BLUE)

# 1b. Missing data per sensor (distribution)
ax2 = fig.add_subplot(gs[0, 1])
missing_pct = df[sensor_cols].isna().mean() * 100
missing_pct_nonzero = missing_pct[missing_pct > 0]
ax2.hist(missing_pct_nonzero, bins=30, color=ORANGE, edgecolor='white', alpha=0.85)
ax2.set_xlabel('Missing % per Sensor')
ax2.set_ylabel('Number of Sensors')
ax2.set_title(f'Data Completeness\n({len(missing_pct_nonzero)} sensors have missing values)', fontweight='bold', color=BLUE)
ax2.axvline(missing_pct_nonzero.median(), color=RED, linestyle='--', linewidth=1.5,
            label=f'Median: {missing_pct_nonzero.median():.1f}%')
ax2.legend(fontsize=9)

# 1c. Sensors with zero variance (useless sensors)
ax3 = fig.add_subplot(gs[0, 2])
sensor_std = df[sensor_cols].std()
zero_var = (sensor_std == 0).sum()
low_var  = ((sensor_std > 0) & (sensor_std < 0.01)).sum()
useful   = (sensor_std >= 0.01).sum()
bars = ax3.bar(['Zero Variance\n(Useless)', 'Low Variance\n(<0.01)', 'Informative\n(≥0.01)'],
               [zero_var, low_var, useful],
               color=[RED, ORANGE, GREEN], edgecolor='white', alpha=0.85)
for bar, val in zip(bars, [zero_var, low_var, useful]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             str(val), ha='center', fontweight='bold', fontsize=11)
ax3.set_ylabel('Number of Sensors')
ax3.set_title('Sensor Signal Quality\n(590 total sensors)', fontweight='bold', color=BLUE)

# 1d. Simulated yield over time (rolling window)
ax4 = fig.add_subplot(gs[1, :])
df_time = df.copy()
df_time['wafer_id'] = range(len(df_time))
df_time['is_fail'] = (df_time['label'] == 1).astype(int)
rolling_yield = (1 - df_time['is_fail'].rolling(window=50, min_periods=10).mean()) * 100

ax4.fill_between(df_time['wafer_id'], rolling_yield, alpha=0.15, color=BLUE)
ax4.plot(df_time['wafer_id'], rolling_yield, color=BLUE, linewidth=1.5, label='Rolling Yield (50-wafer window)')
ax4.axhline(y=rolling_yield.mean(), color=GREEN, linestyle='--', linewidth=1.5,
            label=f'Avg Yield: {rolling_yield.mean():.1f}%')
ax4.axhline(y=90, color=RED, linestyle=':', linewidth=1.5, alpha=0.7,
            label='Warning Threshold: 90%')

# Mark failure clusters
fail_idx = df_time[df_time['is_fail'] == 1]['wafer_id']
ax4.scatter(fail_idx, [88]*len(fail_idx), marker='v', color=RED, s=25, alpha=0.4, label='Individual Failures')

ax4.set_xlabel('Wafer Sequence (Production Order)')
ax4.set_ylabel('Yield Rate (%)')
ax4.set_title('Yield Rate Over Production Run — Rolling 50-Wafer Window', fontweight='bold', color=BLUE)
ax4.legend(loc='lower right', fontsize=9)
ax4.set_ylim(80, 105)
ax4.set_facecolor('#FAFCFF')

plt.savefig('../docs/fig01_yield_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 01 saved.')

In [ ]:
# ── Key EDA Stats ──────────────────────────────────────────────────────────
print('='*55)
print('EDA SUMMARY — Key Numbers for Business Report')
print('='*55)

yield_rate = pass_count / len(df) * 100
defect_rate = fail_count / len(df) * 100

# Cost impact using real industry figures
# Source: PatentPC 2025, SiliconAnalysts 2026
# Using mature node cost of $3,000/wafer as conservative estimate
wafer_cost_low  = 3000   # Mature node (180nm-28nm)
wafer_cost_high = 10000  # Advanced node (7nm-3nm)
monthly_wafer_starts = 5000  # Typical mid-size fab monthly starts

monthly_fails_low  = monthly_wafer_starts * (defect_rate / 100)
monthly_loss_low   = monthly_fails_low * wafer_cost_low
monthly_loss_high  = monthly_fails_low * wafer_cost_high

print(f'Yield Rate              : {yield_rate:.1f}%')
print(f'Defect Rate             : {defect_rate:.1f}%')
print(f'Total Wafers Analyzed   : {len(df):,}')
print(f'Failed Wafers           : {fail_count}')
print(f'Sensors with 0 variance : {zero_var} (can be dropped immediately)')
print(f'Informative sensors     : {useful} / {len(sensor_cols)}')
print()
print('COST IMPACT MODEL (at 5,000 wafer starts/month fab):')
print(f'  Monthly failures      : ~{monthly_fails_low:.0f} wafers')
print(f'  Revenue at risk (low) : ${monthly_loss_low:,.0f}/month (mature node)')
print(f'  Revenue at risk (high): ${monthly_loss_high:,.0f}/month (advanced node)')
print()
print('→ Reducing defect rate by just 10% saves:')
print(f'  ${monthly_loss_low*0.10:,.0f} – ${monthly_loss_high*0.10:,.0f} per month')